In [1]:
import torch
print("CUDA beschikbaar:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA beschikbaar: True
Device: Tesla T4


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

In [3]:
URL = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv"
df = pd.read_csv(URL)
df = df.dropna(subset=["smiles"]).reset_index(drop=True)

print(f"Aantal moleculen: {len(df)}")
print(f"Class balance:\n{df['p_np'].value_counts()}")

Aantal moleculen: 2050
Class balance:
p_np
1    1567
0     483
Name: count, dtype: int64


In [4]:
smiles = df["smiles"].tolist()
labels = df["p_np"].astype(int).tolist()

train_smiles, test_smiles, train_labels, test_labels = train_test_split(
    smiles, labels, test_size=0.1, random_state=42, stratify=labels
)
train_smiles, val_smiles, train_labels, val_labels = train_test_split(
    train_smiles, train_labels, test_size=0.1, random_state=42, stratify=train_labels
)

print(f"Train: {len(train_smiles)} | Val: {len(val_smiles)} | Test: {len(test_smiles)}")

Train: 1660 | Val: 185 | Test: 205


In [5]:
# Verzamel alle unieke characters uit de TRAIN-set
# (niet uit val/test, anders is dat data leakage)
all_chars = set()
for smi in train_smiles:
    all_chars.update(smi)

# Sorteer voor reproduceerbaarheid
sorted_chars = sorted(all_chars)

# Bouw vocab: 0 = padding, 1 = unknown, dan de characters
char_to_idx = {"<PAD>": 0, "<UNK>": 1}
for i, c in enumerate(sorted_chars, start=2):
    char_to_idx[c] = i

idx_to_char = {i: c for c, i in char_to_idx.items()}
vocab_size = len(char_to_idx)

print(f"Vocab size: {vocab_size}")
print(f"Unieke characters: {sorted_chars}")

Vocab size: 41
Unieke characters: ['#', '%', '(', ')', '+', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=', '@', 'B', 'C', 'F', 'H', 'I', 'N', 'O', 'P', 'S', '[', '\\', ']', 'a', 'c', 'l', 'n', 'o', 'r', 's']


In [6]:
lengths = [len(s) for s in train_smiles]
print(f"SMILES lengtes — min: {min(lengths)}, max: {max(lengths)}, "
      f"mean: {np.mean(lengths):.1f}, 95-percentiel: {int(np.percentile(lengths, 95))}")

SMILES lengtes — min: 3, max: 400, mean: 51.8, 95-percentiel: 106


In [7]:
MAX_LENGTH = 200

def encode_smiles(smi, char_to_idx, max_length=MAX_LENGTH):
    """Zet een SMILES-string om in een lijst integers van vaste lengte."""
    # Truncate als hij te lang is
    smi = smi[:max_length]
    # Map elk character naar zijn index, onbekende → <UNK>
    ids = [char_to_idx.get(c, char_to_idx["<UNK>"]) for c in smi]
    # Pad rechts met 0 (=<PAD>) tot max_length
    ids = ids + [char_to_idx["<PAD>"]] * (max_length - len(ids))
    return ids


class SMILES_CNN_Dataset(Dataset):
    def __init__(self, smiles, labels, char_to_idx, max_length=MAX_LENGTH):
        self.smiles = smiles
        self.labels = labels
        self.char_to_idx = char_to_idx
        self.max_length = max_length

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        ids = encode_smiles(self.smiles[idx], self.char_to_idx, self.max_length)
        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }


train_ds = SMILES_CNN_Dataset(train_smiles, train_labels, char_to_idx)
val_ds   = SMILES_CNN_Dataset(val_smiles,   val_labels,   char_to_idx)
test_ds  = SMILES_CNN_Dataset(test_smiles,  test_labels,  char_to_idx)

# Even checken
print(f"Train dataset size: {len(train_ds)}")
print(f"Eerste sample shape: {train_ds[0]['input_ids'].shape}")
print(f"Eerste 30 tokens van eerste sample: {train_ds[0]['input_ids'][:30]}")

Train dataset size: 1660
Eerste sample shape: torch.Size([200])
Eerste 30 tokens van eerste sample: tensor([35, 11, 35, 35, 35, 12, 35,  4, 35,  4, 23,  4, 27, 31, 23, 21, 21, 25,
        33,  4, 23, 23,  5, 35, 13, 35, 35, 35, 35, 35])


In [8]:
class SMILES_CNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_filters=64,
                 kernel_sizes=(3, 5, 7), dropout=0.5, num_classes=2):
        super().__init__()

        # 1) Embedding: token-id → vector van embed_dim
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # 2) Drie parallelle 1D-convoluties met verschillende kernel-sizes
        # Elke kernel kijkt naar een ander "venster" SMILES-tekens.
        # k=3 ziet drielettermotifs zoals "C=O" of "c1c"
        # k=5 en k=7 zien grotere fragmenten
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim,
                      out_channels=num_filters,
                      kernel_size=k,
                      padding=k // 2)
            for k in kernel_sizes
        ])

        # 3) Dropout om overfitting tegen te gaan
        self.dropout = nn.Dropout(dropout)

        # 4) Fully-connected output: alle filter-activaties → 2 classes
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, input_ids, labels=None):
        # input_ids: (batch, seq_len)
        x = self.embedding(input_ids)            # (batch, seq_len, embed_dim)
        x = x.permute(0, 2, 1)                   # (batch, embed_dim, seq_len)
        # Conv1d in PyTorch verwacht channels als 2e dimensie

        # Pas elke conv toe + ReLU + max-pool over de tijd
        conv_outputs = []
        for conv in self.convs:
            c = F.relu(conv(x))                  # (batch, num_filters, seq_len)
            p = F.max_pool1d(c, c.size(2)).squeeze(2)  # (batch, num_filters)
            conv_outputs.append(p)

        # Concat alle filter-outputs samen
        x = torch.cat(conv_outputs, dim=1)       # (batch, num_filters * 3)
        x = self.dropout(x)
        logits = self.fc(x)                      # (batch, 2)

        # Loss berekenen als labels zijn meegegeven (handig voor training)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)

        return {"loss": loss, "logits": logits}


# Instantieer het model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SMILES_CNN(vocab_size=vocab_size).to(device)

# Bekijk hoeveel parameters het model heeft
num_params = sum(p.numel() for p in model.parameters())
print(f"Aantal parameters: {num_params:,}")
print(model)

Aantal parameters: 128,706
SMILES_CNN(
  (embedding): Embedding(41, 128, padding_idx=0)
  (convs): ModuleList(
    (0): Conv1d(128, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): Conv1d(128, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (2): Conv1d(128, 64, kernel_size=(7,), stride=(1,), padding=(3,))
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=192, out_features=2, bias=True)
)


In [9]:
from torch.utils.data import DataLoader

# DataLoaders: zorgen voor batching en (voor train) shuffling
BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)


def evaluate(model, loader, device):
    """Evalueer model op een loader, geeft loss/accuracy/AUC terug."""
    model.eval()
    total_loss = 0.0
    all_logits, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            out = model(input_ids, labels=labels)
            total_loss += out["loss"].item() * input_ids.size(0)
            all_logits.append(out["logits"].cpu())
            all_labels.append(labels.cpu())

    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels).numpy()
    preds = logits.argmax(dim=1).numpy()
    probs = torch.softmax(logits, dim=1)[:, 1].numpy()

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(labels, preds),
        "roc_auc":  roc_auc_score(labels, probs),
    }


def train_model(model, train_loader, val_loader, epochs=30, lr=1e-3, weight_decay=1e-4):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_auc = 0.0
    best_state = None
    history = []

    print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>9} | {'Val Acc':>7} | {'Val AUC':>7}")
    print("-" * 55)

    for epoch in range(1, epochs + 1):
        # ---- Training ----
        model.train()
        epoch_loss = 0.0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            out = model(input_ids, labels=labels)
            out["loss"].backward()
            optimizer.step()

            epoch_loss += out["loss"].item() * input_ids.size(0)

        train_loss = epoch_loss / len(train_loader.dataset)

        # ---- Validation ----
        val_metrics = evaluate(model, val_loader, device)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            **val_metrics,
        })

        print(f"{epoch:>5} | {train_loss:>10.4f} | {val_metrics['loss']:>9.4f} "
              f"| {val_metrics['accuracy']:>7.4f} | {val_metrics['roc_auc']:>7.4f}")

        # Sla het beste model op (op basis van val AUC)
        if val_metrics["roc_auc"] > best_val_auc:
            best_val_auc = val_metrics["roc_auc"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    # Laad het beste model terug
    model.load_state_dict(best_state)
    print(f"\nBeste val AUC: {best_val_auc:.4f}")
    return history

In [10]:
# Voor reproduceerbaarheid (best wel belangrijk voor een rapport!)
torch.manual_seed(42)
np.random.seed(42)

# Herstart het model met dezelfde seed zodat de initialisatie gelijk is
model = SMILES_CNN(vocab_size=vocab_size).to(device)

history = train_model(model, train_loader, val_loader, epochs=30)

Epoch | Train Loss |  Val Loss | Val Acc | Val AUC
-------------------------------------------------------
    1 |     0.4808 |    0.3202 |  0.8486 |  0.9209
    2 |     0.3462 |    0.2787 |  0.8865 |  0.9313
    3 |     0.2940 |    0.2712 |  0.8811 |  0.9318
    4 |     0.2672 |    0.2961 |  0.8649 |  0.9297
    5 |     0.2449 |    0.2659 |  0.9027 |  0.9338
    6 |     0.2392 |    0.2595 |  0.8919 |  0.9342
    7 |     0.2259 |    0.2846 |  0.8865 |  0.9325
    8 |     0.2161 |    0.2795 |  0.8919 |  0.9381
    9 |     0.2079 |    0.2813 |  0.8703 |  0.9310
   10 |     0.1957 |    0.3221 |  0.8811 |  0.9304
   11 |     0.1742 |    0.2912 |  0.8649 |  0.9257
   12 |     0.2014 |    0.2882 |  0.8865 |  0.9297
   13 |     0.1756 |    0.3126 |  0.8811 |  0.9260
   14 |     0.1596 |    0.3141 |  0.8811 |  0.9212
   15 |     0.1624 |    0.3292 |  0.9027 |  0.9242
   16 |     0.1393 |    0.3046 |  0.8757 |  0.9263
   17 |     0.1489 |    0.3089 |  0.8973 |  0.9328
   18 |     0.1410 |    0.

In [11]:
test_metrics = evaluate(model, test_loader, device)
print("=== Testresultaten CNN ===")
for k, v in test_metrics.items():
    print(f"{k:10s}: {v:.4f}")

# Voorspellingen voor confusion matrix
model.eval()
all_preds, all_probs, all_labels = [], [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        out = model(input_ids)
        logits = out["logits"]
        all_preds.append(logits.argmax(dim=1).cpu().numpy())
        all_probs.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
        all_labels.append(batch["labels"].numpy())

y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_labels)

cm = confusion_matrix(y_true, y_pred)
print("\nConfusion matrix:")
print("                 voorspeld 0   voorspeld 1")
print(f"  werkelijk 0:    {cm[0,0]:5d}        {cm[0,1]:5d}")
print(f"  werkelijk 1:    {cm[1,0]:5d}        {cm[1,1]:5d}")

print("\nPer-klasse metrics:")
print(classification_report(y_true, y_pred, target_names=["geen BBB (0)", "wel BBB (1)"]))

=== Testresultaten CNN ===
loss      : 0.2084
accuracy  : 0.9073
roc_auc   : 0.9655

Confusion matrix:
                 voorspeld 0   voorspeld 1
  werkelijk 0:       34           14
  werkelijk 1:        5          152

Per-klasse metrics:
              precision    recall  f1-score   support

geen BBB (0)       0.87      0.71      0.78        48
 wel BBB (1)       0.92      0.97      0.94       157

    accuracy                           0.91       205
   macro avg       0.89      0.84      0.86       205
weighted avg       0.91      0.91      0.90       205

